# E-Commerce Customer Analytics
### Complete Data Analytics Project

**Workflow:** Raw Dataset → Data Cleaning → Validation → EDA → KPI Analysis → Customer Segmentation → Business Insights

> AI/ML is intentionally excluded from this version. The notebook focuses on descriptive and diagnostic analytics.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load the raw dataset

Place `synthetic_ecommerce_churn_dataset.csv` in the same folder as this notebook, or update `RAW_FILE` below.


In [ ]:
# Update this path if your raw CSV is stored elsewhere
RAW_FILE = "synthetic_ecommerce_churn_dataset.csv"

df = pd.read_csv(RAW_FILE)

print("Raw shape:", df.shape)
display(df.head())


## 3. Initial data understanding

We inspect structure, data types, missing values, duplicates, and basic statistics before cleaning.


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe(include="all").T)


## 4. Data cleaning

Cleaning steps:
- Standardize column names
- Remove leading/trailing whitespace from text
- Convert numeric columns to numeric datatype
- Convert date columns to datetime
- Remove exact duplicate rows
- Validate business ranges
- Impute numeric missing values with median
- Impute categorical missing values with mode


In [ ]:
# 4.1 Standardize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

# 4.2 Strip whitespace from text columns
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# 4.3 Convert numeric columns
numeric_cols = [
    "age", "avg_order_value", "total_orders",
    "is_fraudulent", "email_open_rate",
    "loyalty_score", "churn_risk"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 4.4 Convert date columns
date_cols = ["last_purchase", "customer_since"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 4.5 Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", before - len(df))


In [ ]:
# 4.6 Validate business ranges
range_rules = {
    "age": df["age"].between(0, 120),
    "avg_order_value": df["avg_order_value"].ge(0),
    "total_orders": df["total_orders"].ge(0),
    "email_open_rate": df["email_open_rate"].between(0, 100),
    "loyalty_score": df["loyalty_score"].between(0, 100),
    "churn_risk": df["churn_risk"].between(0, 1),
    "is_fraudulent": df["is_fraudulent"].isin([0, 1])
}

for col, valid_mask in range_rules.items():
    invalid_count = (~valid_mask.fillna(False)).sum()
    if invalid_count:
        df.loc[~valid_mask.fillna(False), col] = np.nan
    print(f"{col}: invalid values handled = {invalid_count}")


In [ ]:
# 4.7 Impute missing numeric values with median
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# 4.8 Impute missing categorical values with mode
categorical_cols = [
    "customer_id", "gender", "country",
    "preferred_category"
]

for col in categorical_cols:
    if df[col].isna().any():
        mode_value = df[col].mode(dropna=True)
        if len(mode_value):
            df[col] = df[col].fillna(mode_value.iloc[0])

print("Cleaning completed.")


## 5. Final cleaning validation

The following checks confirm that the working dataset is ready for analysis.


In [ ]:
print("Cleaned shape:", df.shape)
print("\nMissing values after cleaning:")
display(df.isna().sum())

print("Total missing values:", int(df.isna().sum().sum()))
print("Duplicate rows after cleaning:", int(df.duplicated().sum()))

print("\nData types after cleaning:")
display(df.dtypes)


## 6. Save the cleaned dataset

This creates a reusable cleaned CSV for SQL and Power BI work.


In [ ]:
CLEAN_FILE = "ecommerce_customer_cleaned.csv"
df.to_csv(CLEAN_FILE, index=False)

print(f"Saved cleaned dataset to: {CLEAN_FILE}")


## 7. Basic EDA


In [ ]:
display(df.head(10))
display(df.describe().T)


In [ ]:
# Numeric distributions
df[numeric_cols].hist(figsize=(14, 10), bins=25)
plt.tight_layout()
plt.show()


## 8. KPI Analysis


In [ ]:
total_customers = df["customer_id"].nunique()
total_orders = df["total_orders"].sum()
avg_order_value = df["avg_order_value"].mean()
avg_loyalty = df["loyalty_score"].mean()
fraud_rate = df["is_fraudulent"].mean() * 100

kpis = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Total Orders",
        "Average Order Value",
        "Average Loyalty Score",
        "Fraud Rate (%)"
    ],
    "Value": [
        total_customers,
        total_orders,
        avg_order_value,
        avg_loyalty,
        fraud_rate
    ]
})

display(kpis)


## 9. Customer Value Segmentation

Customer value is derived as:

**Customer Value = Average Order Value × Total Orders**

Segments are created using the 33rd and 67th percentiles. This is a descriptive business segmentation, not machine learning.


In [ ]:
df["customer_value"] = df["avg_order_value"] * df["total_orders"]

q33 = df["customer_value"].quantile(0.33)
q67 = df["customer_value"].quantile(0.67)

df["customer_segment"] = pd.cut(
    df["customer_value"],
    bins=[-np.inf, q33, q67, np.inf],
    labels=["Low Value", "Mid Value", "High Value"]
)

segment_summary = (
    df.groupby("customer_segment", observed=True)
      .agg(
          customers=("customer_id", "nunique"),
          avg_customer_value=("customer_value", "mean"),
          avg_orders=("total_orders", "mean"),
          avg_order_value=("avg_order_value", "mean"),
          avg_loyalty=("loyalty_score", "mean")
      )
      .reset_index()
)

display(segment_summary)


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="customer_segment")
plt.title("Customer Segments")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


## 10. Preferred Category Analysis


In [ ]:
category_summary = (
    df.groupby("preferred_category")
      .agg(
          customers=("customer_id", "nunique"),
          total_orders=("total_orders", "sum"),
          avg_order_value=("avg_order_value", "mean"),
          avg_loyalty=("loyalty_score", "mean")
      )
      .sort_values("total_orders", ascending=False)
)

display(category_summary)


In [ ]:
plt.figure(figsize=(10, 5))
category_summary["total_orders"].sort_values().plot(kind="barh")
plt.title("Total Orders by Preferred Category")
plt.xlabel("Total Orders")
plt.ylabel("Preferred Category")
plt.tight_layout()
plt.show()


## 11. Country-wise Analysis


In [ ]:
country_summary = (
    df.groupby("country")
      .agg(
          customers=("customer_id", "nunique"),
          total_orders=("total_orders", "sum"),
          avg_order_value=("avg_order_value", "mean")
      )
      .sort_values("total_orders", ascending=False)
)

display(country_summary.head(15))


In [ ]:
top_countries = country_summary.head(10).sort_values("total_orders")

plt.figure(figsize=(10, 6))
top_countries["total_orders"].plot(kind="barh")
plt.title("Top 10 Countries by Total Orders")
plt.xlabel("Total Orders")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


## 12. Loyalty & Engagement Analysis


In [ ]:
loyalty_summary = df[[
    "loyalty_score",
    "email_open_rate",
    "avg_order_value",
    "total_orders"
]].corr(numeric_only=True)

display(loyalty_summary)


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(loyalty_summary, annot=True, fmt=".2f")
plt.title("Loyalty & Engagement Correlation")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="loyalty_score",
    y="customer_value",
    alpha=0.5
)
plt.title("Loyalty Score vs Customer Value")
plt.xlabel("Loyalty Score")
plt.ylabel("Customer Value")
plt.tight_layout()
plt.show()


## 13. Fraud Indicator Analysis

`is_fraudulent` is treated as a dataset-provided indicator. The analysis describes the observed records; it does not infer fraud beyond that field.


In [ ]:
fraud_summary = (
    df.groupby("is_fraudulent")
      .agg(
          customers=("customer_id", "nunique"),
          avg_order_value=("avg_order_value", "mean"),
          avg_orders=("total_orders", "mean"),
          avg_loyalty=("loyalty_score", "mean")
      )
      .reset_index()
)

display(fraud_summary)


## 14. Business Questions


In [ ]:
# Q1: Which customer segment has the highest average customer value?
display(
    segment_summary.sort_values("avg_customer_value", ascending=False)
)

# Q2: Which categories have the highest order volume?
display(category_summary.sort_values("total_orders", ascending=False).head(10))

# Q3: Which countries have the highest order volume?
display(country_summary.head(10))

# Q4: How do loyalty and customer value relate?
print("Correlation between loyalty score and customer value:",
      df["loyalty_score"].corr(df["customer_value"]))

# Q5: What percentage of records are marked fraudulent?
print("Fraud indicator rate (%):", round(df["is_fraudulent"].mean() * 100, 2))


## 15. Final Business Insights

Use the outputs above to write project-specific observations such as:
- customer concentration by segment,
- categories with higher order volume,
- countries contributing more orders,
- relationship between loyalty and customer value,
- observed fraud-indicator rate.

These statements should be based on the values produced by the notebook rather than assumed in advance.


In [ ]:
print("Project analysis completed successfully.")
print(f"Customers: {total_customers:,}")
print(f"Total orders: {total_orders:,.0f}")
print(f"Average order value: {avg_order_value:,.2f}")
print(f"Average loyalty score: {avg_loyalty:,.2f}")
print(f"Fraud indicator rate: {fraud_rate:.2f}%")
